# Phase 3.7: Model Explainability (SHAP)
In this notebook, we interpret the best performing tree-based model (XGBoost) using SHAP values to understand feature importance and interaction effects.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
import sys
import shap
import matplotlib.pyplot as plt

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path: sys.path.append(project_root)

from src.utils.train_utils import temporal_split
from src.utils.eval_utils import generate_shap_plots

plt.style.use('ggplot')
os.makedirs("../visualization/model_performance/shap", exist_ok=True)

## 1. Load Data and Best Model

In [ ]:
df = pd.read_csv("../data/processed/dynamic_pricing_processed.csv")
df_train, df_val, df_test = temporal_split(df)

TARGET = "Historical_Cost_of_Ride"
X_train = df_train.drop(columns=[TARGET]).select_dtypes(include=[np.number])
X_test = df_test.drop(columns=[TARGET]).select_dtypes(include=[np.number])

model = joblib.load("../models/xgboost_best.pkl")
print(f"Loaded model: {type(model).__name__}")

## 2. SHAP Summary (Global Importance)

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_test)

# Beeswarm Plot
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title("SHAP Beeswarm: Impact on Price")
plt.savefig("../visualization/model_performance/shap/summary_beeswarm.png")
plt.show()

# Bar Plot
shap.plots.bar(shap_values, max_display=15, show=False)
plt.title("SHAP Bar: Mean |SHAP Value|")
plt.savefig("../visualization/model_performance/shap/summary_bar.png")
plt.show()

## 3. Feature Dependence Plots

In [ ]:
top_features = ["demand_supply_ratio", "Expected_Ride_Duration", "hour_sin", "driver_deficit", "location_score"]

for feat in top_features:
    shap.plots.scatter(shap_values[:, feat], color=shap_values[:, "demand_supply_ratio"], show=False)
    plt.title(f"Dependence: {feat}")
    plt.savefig(f"../visualization/model_performance/shap/dependence_{feat}.png")
    plt.show()

## 4. Local Explanations (Waterfall)

In [ ]:
sample_indices = [0, 50, 100]
for idx in sample_indices:
    shap.plots.waterfall(shap_values[idx], show=False)
    plt.title(f"Waterfall Plot: Sample {idx}")
    plt.savefig(f"../visualization/model_performance/shap/waterfall_sample_{idx}.png")
    plt.show()